In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os

def find_file(filename, search_path):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

# 1. Automatically locate the file anywhere in your project
filename = "kenya.csv"
# We search starting from two levels up to be safe
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
file_path = find_file(filename, project_root)

if file_path:
    print(f"Found it! Loading from: {file_path}")
    df = pd.read_csv(file_path)
else:
    # Last resort: list everything so we can see where it is
    print("Files in current folder:", os.listdir("."))
    raise FileNotFoundError(f"I searched everywhere but couldn't find {filename}. Is it spelled exactly like that?")

# 2. Add Country & Clean NASA Sentinel Values
df['Country'] = 'Kenya'
df.replace(-999, np.nan, inplace=True)

# 3. Date Parsing
# NASA uses YEAR and Day of Year (DOY). 
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['Date'].dt.month

# 4. Outlier Detection (Z-score > 3)
cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
df_numeric = df[cols_to_check].dropna()
z_scores = np.abs(stats.zscore(df_numeric))

print("\n--- Success! Kenya Data Profiled ---")
print(f"Total rows: {len(df)}")
print(f"Outlier Counts (|Z| > 3):\n{(z_scores > 3).sum()}")

# 5. Export Clean Data to a safe spot
df.to_csv("kenya_clean.csv", index=False)
df.head()

Found it! Loading from: c:\Users\HP\Desktop\10acadamey\week0\-climate-challenge-week0\.github\workflows\kenya.csv

--- Success! Kenya Data Profiled ---
Total rows: 4108
Outlier Counts (|Z| > 3):
112


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88,Kenya,2015-01-01,1
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85,Kenya,2015-01-02,1
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65,Kenya,2015-01-03,1
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60,Kenya,2015-01-04,1
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64,Kenya,2015-01-05,1
